# Three way chat bot

In [ ]:
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display
import json

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")


In [ ]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

openai = OpenAI()

ollama_url = "http://localhost:11434/v1"

ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [ ]:
# Let's make a conversation between GPT-4.1-mini and Claude-haiku-4.5
# We're using cheap versions of models so the costs will be minimal

gpt_mini_model = "gpt-4.1-mini" # Blake
gpt_nano_model = "gpt-5-nano" # Charlie
gpt_mini_model_2 = "gpt-4.1-mini" # Alex

gpt_blake_system = "Your name is Blake. You are a chatbot who is very argumentative; \
you disagree with anything in the conversation and you challenge everything, in a snarky way."
    
gpt_charlie_system = "Your name is Charlie. You are a chatbot who is very agreeable; you agree with anything in the conversation and you support everything, in a very nice way."

gpt_alex_system = "Your name is Alex. You are a chatbot who is very curious; you ask questions and seek to understand the perspectives of others. \
    You are in a conversation with Blake and Charlie."

def get_gpt_alex_user_prompt(conversation):
    gpt_alex_user = f"""
    You are Alex, in conversation with Blake and Charlie.
    The conversation so far is as follows:
    {conversation}
    Now with this, respond with what you would like to say next, as Alex.
    """
    return gpt_alex_user

blake_messages = ["Hi there"]
charlie_messages = ["Hi"]
alex_messages = ["Hello"]

In [ ]:
def call_blake():
    messages = [{"role": "system", "content": gpt_blake_system}]
    for blake, charlie in zip(blake_messages, charlie_messages):
        messages.append({"role": "assistant", "content": blake})
        messages.append({"role": "user", "content": charlie})
    # print("GPT Messages:", messages)
    response = openai.chat.completions.create(model=gpt_mini_model, messages=messages)
    return response.choices[0].message.content

In [ ]:
call_blake()

In [ ]:
def call_charlie():
    messages = [{"role": "system", "content": gpt_charlie_system}]
    for blake, charlie in zip(blake_messages, charlie_messages):
        messages.append({"role": "assistant", "content": charlie})
        messages.append({"role": "user", "content": blake})
    # print("GPT Messages:", messages)
    response = openai.chat.completions.create(model=gpt_nano_model, messages=messages)
    return response.choices[0].message.content

In [ ]:
call_charlie()

In [ ]:
def get_conversation_history():
    conversation = {}
    for i, (blake, charlie, alex) in enumerate(zip(blake_messages, charlie_messages, alex_messages)):
        conversation[f"Blake: {i}"] = blake
        conversation[f"Charlie: {i}"] = charlie
        conversation[f"Alex: {i}"] = alex
    return json.dumps(conversation)

In [ ]:
get_conversation_history()

In [ ]:
def call_alex():
    conversation = get_conversation_history()
    messages = [{"role": "system", "content": gpt_alex_system}]
    user_prompt = get_gpt_alex_user_prompt(conversation)
    messages.append({"role": "user", "content": user_prompt})
    response = openai.chat.completions.create(model=gpt_mini_model_2, messages=messages)
    return response.choices[0].message.content

In [ ]:
call_alex()

In [ ]:
display(Markdown(f"### Blake:\n{blake_messages[0]}\n"))
display(Markdown(f"### Charlie:\n{charlie_messages[0]}\n"))
display(Markdown(f"### Alex:\n{alex_messages[0]}\n"))

for i in range(5):
    blake_next = call_blake()
    display(Markdown(f"### Blake:\n{blake_next}\n"))
    blake_messages.append(blake_next)

    charlie_next = call_charlie()
    display(Markdown(f"### Charlie:\n{charlie_next}\n"))
    charlie_messages.append(charlie_next)

    alex_next = call_alex()
    display(Markdown(f"### Alex:\n{alex_next}\n"))
    alex_messages.append(alex_next)